In [3]:
from pathlib import Path
import re
import pandas as pd
from pandas import DataFrame

from helpers import rename_patient
from written_thesis.helpers import PRETTY_FEATURE_NAMES_MAP

In [14]:
in_path = Path('~/thesis_files/statistical_results/.model_feature_qualifications.pkl').expanduser()

In [15]:
orig = pd.read_pickle(in_path)
orig.head()

seizures                   CNN              \
metric                 p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                    
competition-1 corrcoef      0.404490       False  0.610688       False   
              acfw_D        0.000561        True  0.610688       False   
              acfw_P        0.008206        True  0.610688       False   
              var_D         0.607431       False  0.610688       False   
              var_P         0.032674        True  0.610688       False   

                                                                              \
metric                  p_rayleigh_bh significant p_perm_bh    met qualified   
patient       feature                                                          
competition-1 corrcoef  2.284097e-106        True  0.973005   True     False   
              acfw_D     1.311392e-54        True  0.001000  False     False   
              acfw_P     1.048648e-01       False  0.006999  False     False   
              var_D      4.328165e-17        True  0.219648   True     False   
              var_P      2.566058e-58        True  0.001000  False     False   

                       ensemble                                         \
metric                  roc_auc roc_auc_met  p_rayleigh_bh significant   
patient       feature                                                    
competition-1 corrcoef  0.59268       False  1.334214e-138        True   
              acfw_D    0.59268       False   5.955082e-81        True   
              acfw_P    0.59268       False   2.067303e-02        True   
              var_D     0.59268       False   2.161849e-19        True   
              var_P     0.59268       False   1.917672e-78        True   

                                                   
metric                 p_perm_bh    met qualified  
patient       feature                              
competition-1 corrcoef  0.984603   True     False  
              acfw_D    0.000750  False     False  
              acfw_P    0.002999  False     False  
              var_D     0.258179   True     False  
              var_P     0.001000  False     False

In [16]:
cnn = orig.drop(columns=['ensemble']).copy()
ens = orig.drop(columns=['CNN']).copy()

In [17]:
cnn_qual = cnn[cnn[('CNN', 'qualified')]]
cnn_qual

seizures                   CNN              \
metric                p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                   
competition-2 acfw_D       0.012896        True  0.796782        True   

                                                                            
metric                 p_rayleigh_bh significant p_perm_bh   met qualified  
patient       feature                                                       
competition-2 acfw_D   4.843678e-123        True   0.10138  True      True

In [18]:
ens_qual = ens[ens[('ensemble', 'qualified')]]
ens_qual

seizures              ensemble              \
metric               p_rayleigh_bh significant   roc_auc roc_auc_met   
patient      feature                                                   
U002-DE01-17 Alpha_P      0.003536        True  0.787532        True   
             Beta_P       0.003299        True  0.787532        True   

                                                                          
metric               p_rayleigh_bh significant p_perm_bh   met qualified  
patient      feature                                                      
U002-DE01-17 Alpha_P  4.651312e-11        True  0.243497  True      True  
             Beta_P   4.336590e-07        True  0.613377  True      True

In [31]:
def apply_transformations(df: DataFrame, model: str) -> DataFrame:
    t = df.copy()

    # Abbreviate patients
    idx_df = t.index.to_frame(index=False)
    idx_df['patient'] = idx_df['patient'].map(rename_patient)
    t.index = pd.MultiIndex.from_frame(idx_df)

    # Drop columns that indicate whether criteria were met, since we only show combinations where all criteria were met.
    t = t.drop(columns=[('seizures', 'significant')] + [(model, col) for col in
                                                        ['significant', 'roc_auc_met', 'met', 'qualified']])

    # Insert Model Type as a column
    t = t.rename(columns={'ensemble': 'model', 'CNN': 'model'})
    t.insert(t.columns.get_loc(('model', 'roc_auc')), ('model', 'type'), model)

    # Rename
    t = t.rename_axis(index={'patient': r'\textbf{Patient}', 'feature': r'\textbf{Feature}'},
                      columns={'metric': None})
    t = t.rename(
        columns={
            'seizures': r'\textbf{Seizures}',
            'model': r'\textbf{Model}',
            'type': r'\textbf{Type}',

            'p_rayleigh_bh': r'$\boldsymbol{q_{\mathrm{Rayleigh}}}$',
            'significant': r'\boldsymbol{$<0.05$}',

            'roc_auc': r'\textbf{ROC AUC}',
            'roc_auc_met': r'$\boldsymbol{\geq 0.75$}',

            'p_perm_bh': r'$\boldsymbol{q_{\mathrm{Permutation}}}$}',
            'met': r'\boldsymbol{$\geq 0.05$}',
        },
        index=PRETTY_FEATURE_NAMES_MAP,
    )

    return t


cnn_t = apply_transformations(cnn_qual, 'CNN')
ens_t = apply_transformations(ens_qual, 'ensemble')

combined = pd.concat([cnn_t, ens_t])
combined

\textbf{Seizures}  \
                                  $\boldsymbol{q_{\mathrm{Rayleigh}}}$   
\textbf{Patient} \textbf{Feature}                                        
C02              ACFW (D)                                     0.012896   
U17              Alpha (P)                                    0.003536   
                 Beta (P)                                     0.003299   

                                  \textbf{Model}                   \
                                   \textbf{Type} \textbf{ROC AUC}   
\textbf{Patient} \textbf{Feature}                                   
C02              ACFW (D)                    CNN         0.796782   
U17              Alpha (P)              ensemble         0.787532   
                 Beta (P)               ensemble         0.787532   

                                                                        \
                                  $\boldsymbol{q_{\mathrm{Rayleigh}}}$   
\textbf{Patient} \textbf{Feature}                                        
C02              ACFW (D)                                4.843678e-123   
U17              Alpha (P)                                4.651312e-11   
                 Beta (P)                                 4.336590e-07   

                                                                            
                                  $\boldsymbol{q_{\mathrm{Permutation}}}$}  
\textbf{Patient} \textbf{Feature}                                           
C02              ACFW (D)                                         0.101380  
U17              Alpha (P)                                        0.243497  
                 Beta (P)                                         0.613377

In [32]:
l = combined.to_latex(
    float_format='%.3f',
    escape=False,
    multicolumn=True,
    multicolumn_format='c',
    column_format='llrrrrr',
)
# l = l.replace(r'\bottomrule'+'\n', '')
# l = re.sub(r'(toprule|midrule|bottomrule)', 'hline', l)

print(l)

\begin{tabular}{llrrrrr}
\toprule
 &  & \textbf{Seizures} & \multicolumn{4}{c}{\textbf{Model}} \\
 &  & $\boldsymbol{q_{\mathrm{Rayleigh}}}$ & \textbf{Type} & \textbf{ROC AUC} & $\boldsymbol{q_{\mathrm{Rayleigh}}}$ & $\boldsymbol{q_{\mathrm{Permutation}}}$} \\
\textbf{Patient} & \textbf{Feature} &  &  &  &  &  \\
\midrule
C02 & ACFW (D) & 0.013 & CNN & 0.797 & 0.000 & 0.101 \\
\cline{1-7}
\multirow[t]{2}{*}{U17} & Alpha (P) & 0.004 & ensemble & 0.788 & 0.000 & 0.243 \\
 & Beta (P) & 0.003 & ensemble & 0.788 & 0.000 & 0.613 \\
\cline{1-7}
\bottomrule
\end{tabular}

